# GluonTS

This example notebook demonstrates the compatibility of HierarchicalForecast's reconciliation methods with popular machine-learning libraries, specifically [GluonTS](https://ts.gluon.ai/stable/). 

The notebook utilizes the GluonTS DeepAREstimator to create base forecasts for the TourismLarge Hierarchical Dataset. We make the base forecasts compatible with HierarchicalForecast's reconciliation functions via the `samples_to_quantiles_df` utility function that transforms GluonTS' output forecasts into a compatible data frame format. After that, we use HierarchicalForecast to reconcile the base predictions.

**References**<br>
- [David Salinas, Valentin Flunkert, Jan Gasthaus, Tim Januschowski (2020). "DeepAR: Probabilistic forecasting with autoregressive recurrent networks". International Journal of Forecasting.](https://www.sciencedirect.com/science/article/pii/S0169207019301888)<br>
- [Alexander Alexandrov et. al (2020). "GluonTS: Probabilistic and Neural Time Series Modeling in Python". Journal of Machine Learning Research.](https://www.jmlr.org/papers/v21/19-820.html)<br>

You can run these experiments using CPU or GPU with Google Colab.

<a href="https://colab.research.google.com/github/Nixtla/hierarchicalforecast/blob/main/nbs/examples/HierarchicalForecast-GluonTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Installing packages

In [ ]:
%%capture
!pip install mxnet-cu112

In [ ]:
import mxnet as mx

assert mx.context.num_gpus()>0

In [ ]:
%%capture
!pip install gluonts
!pip install datasetsforecast
!pip install git+https://github.com/Nixtla/hierarchicalforecast.git

In [ ]:
import numpy as np
import pandas as pd

from datasetsforecast.hierarchical import HierarchicalData

from gluonts.mx.trainer import Trainer
from gluonts.dataset.pandas import PandasDataset
from gluonts.mx.model.deepar import DeepAREstimator

from hierarchicalforecast.methods import BottomUp, MinTrace
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.evaluation import scaled_crps
from hierarchicalforecast.utils import samples_to_quantiles_df

## 2. Load hierarchical dataset


This detailed Australian Tourism Dataset comes from the National Visitor Survey, managed by the Tourism Research Australia, it is composed of 555 monthly series from 1998 to 2016, it is organized geographically, and purpose of travel. The natural geographical hierarchy comprises seven states, divided further in 27 zones and 76 regions. The purpose of travel categories are holiday, visiting friends and relatives (VFR), business and other. The MinT (Wickramasuriya et al., 2019), among other hierarchical forecasting studies has used the dataset it in the past. The dataset can be accessed in the [MinT reconciliation webpage](https://robjhyndman.com/publications/mint/), although other sources are available.

| Geographical Division | Number of series per division | Number of series per purpose | Total |
|          ---          |               ---             |              ---             |  ---  |
|  Australia            |              1                |               4              |   5   |
|  States               |              7                |              28              |  35   |
|  Zones                |             27                |              108             |  135  |
|  Regions              |             76                |              304             |  380  |
|  Total                |            111                |              444             |  555  |


In [ ]:
dataset = 'TourismLarge'
Y_df, S_df, tags = HierarchicalData.load(directory = "./data", group=dataset)
Y_df['ds'] = pd.to_datetime(Y_df['ds'])

In [ ]:
def sort_hier_df(Y_df, S_df):
    # sorts unique_id lexicographically
    Y_df.unique_id = Y_df.unique_id.astype('category')
    Y_df.unique_id = Y_df.unique_id.cat.set_categories(S_df.index)
    Y_df = Y_df.sort_values(by=['unique_id', 'ds'])
    return Y_df

Y_df = sort_hier_df(Y_df, S_df)

In [ ]:
horizon = 12

Y_test_df = Y_df.groupby('unique_id').tail(horizon)
Y_train_df = Y_df.drop(Y_test_df.index)
Y_train_df

In [ ]:
ds = PandasDataset.from_long_dataframe(Y_train_df, target="y", item_id="unique_id")

## 3. Fit and Predict Model


In [ ]:
estimator = DeepAREstimator(
    freq="M",
    prediction_length=horizon,
    trainer=Trainer(ctx = mx.context.gpu(),
                    epochs=20),
)
predictor = estimator.train(ds)

forecast_it = predictor.predict(ds, num_samples=1000)

forecasts = list(forecast_it)
forecasts = np.array([arr.samples for arr in forecasts])
forecasts.shape

## 4. Reconciliation


In [ ]:
level = np.arange(1, 100, 2)

#transform the output of DeepAREstimator to a form that is compatible with HierarchicalForecast
quantiles, forecast_df = samples_to_quantiles_df(samples=forecasts, 
                               unique_ids=S_df.index, 
                               dates=Y_test_df['ds'].unique(), 
                               level=level,
                               model_name='DeepAREstimator')

#reconcile forecasts
reconcilers = [
    BottomUp(),
    MinTrace('ols')
]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)

forecast_rec = hrec.reconcile(Y_hat_df=forecast_df, S=S_df, tags=tags, level=level)

In [ ]:
forecast_rec

## 5. Evaluation

To evaluate we use a scaled variation of the CRPS, as proposed by Rangapuram (2021), to measure the accuracy of predicted quantiles `y_hat` compared to the observation `y`.

$$ \mathrm{sCRPS}(\hat{F}_{\tau}, \mathbf{y}_{\tau}) = \frac{2}{N} \sum_{i}
\int^{1}_{0}
\frac{\mathrm{QL}(\hat{F}_{i,\tau}, y_{i,\tau})_{q}}{\sum_{i} | y_{i,\tau} |} dq $$

As you can see, HierarchicalForecast results improve on the results of specialized algorithms like [HierE2E](https://proceedings.mlr.press/v139/rangapuram21a.html).

In [ ]:
rec_model_names = ['DeepAREstimator/MinTrace_method-ols', 'DeepAREstimator/BottomUp']

quantiles = np.array(quantiles[1:]) #remove first quantile (median)
n_quantiles = len(quantiles)
n_series = len(S_df)

for name in rec_model_names:
    quantile_columns = [col for col in forecast_rec.columns if (name+'-') in col]
    y_rec  = forecast_rec[quantile_columns].values 
    y_test = Y_test_df['y'].values

    y_rec  = y_rec.reshape(n_series, horizon, n_quantiles)
    y_test = y_test.reshape(n_series, horizon)
    scrps  = scaled_crps(y=y_test, y_hat=y_rec, quantiles=quantiles)
    print("{:<40} {:.5f}".format(name+":", scrps))